In [ ]:
# Import Basic Packgaes 
import numpy as np
import pandas as pd
import datetime
import statsmodels as sm
import itertools
import glob
import os
from scipy.signal import argrelextrema
from scipy.signal import find_peaks, peak_widths
import scipy as sp

# Data Visualization
import matplotlib.pyplot as plt
import seaborn as sns # advanced vizs
from pandas.plotting import lag_plot

# Ignore Warnings
import warnings
warnings.filterwarnings("ignore")

# Set Color Palettes
palette1 = itertools.cycle(sns.color_palette(palette='Set1'))
palette2 = itertools.cycle(sns.color_palette(palette='Set2'))

In [ ]:
# Load the csv files

df = pd.read_csv("D:\\TDMS\\2.P1-100Hrs to 800hrs\\100 to 200 Hrs_Freq,Amp\\p1_lt_2023_04_07_clean.csv")
df.head()

In [ ]:
clean_df = df[df['P1-Air-Supply-pressure'] > 93]

In [ ]:
clean_df.reset_index(inplace=True, drop=True)

In [ ]:
clean_df['newtime'] = pd.date_range('2023-04-07 00:00:00.000', freq='30ms', periods=len(clean_df))

In [ ]:
clean_df['newtime'] =pd.to_datetime(clean_df['newtime'])

In [ ]:
clean_df.set_index(['newtime'], inplace=True)

In [ ]:
# Define FFT parameters
dt = 0.03
f1, f2 = 1000, 1000  # Hz
diff = 100

In [ ]:
# Define the time interval for slicing
start_time = clean_df.index[0]
end_time = clean_df.index[-1]
time_interval = pd.Timedelta(hours=1)

In [ ]:
current_time = start_time
while current_time <= end_time:
    next_time = current_time + time_interval

    # Slice the DataFrame based on the specified time range
    sliced_df = clean_df.iloc[(clean_df.index >= current_time) & (clean_df.index <= next_time)]

    # Perform FFT analysis only if there is data in the sliced DataFrame
    n = len(sliced_df)
    if n > 0:
        t = sliced_df.index
        s = sliced_df['P1-Air-Supply-pressure']

        # FFT
        s -= s.mean()
        fr = np.fft.rfftfreq(n, dt)
        fou = np.fft.rfft(s)

        # Plot and save the FFT plot
        plt.figure(figsize=(28, 10))
        plt.subplot(211)
        plt.plot(t, s)
        plt.title('Data with Noise')

        plt.subplot(212)
        plt.plot(np.fft.fftshift(fr), np.fft.fftshift(np.abs(fou)))
        plt.xticks(np.arange(min(np.fft.fftshift(fr)), max(np.fft.fftshift(fr)), 0.3))
        plt.ylim(0, 40000)
        plt.title('Spectrum of Noisy Data')

        # Save the plot with a meaningful filename
        filename = os.path.join("D:\\New folder", f'{current_time.strftime("%H-%M-%S")}_{next_time.strftime("%H-%M-%S")}_FFT.png')
        plt.savefig(filename)
        plt.close()
            #fr_list = [np.fft.fftshift(fr)]
            #amp_list = [np.fft.fftshift(np.abs(fou))]
            #fourier_df = pd.DataFrame(data=[fr_list, amp_list],columns=['frequency','amplidute'])
            #print(fourier_df.loc[fourier_df['amplidute'].idxmax()])

    else:
        # Handle the case where the DataFrame is empty (e.g., print a message)
        print(f"No data in the time range {current_time} - {next_time}")

    current_time = next_time

In [ ]:
freq_amp_dfs = []

In [ ]:
current_time = start_time
while current_time <= end_time:
    
    next_time = current_time + time_interval
    
    # Slice the DataFrame based on the specified time range
    sliced_df = clean_df.iloc[(clean_df.index >= current_time) & (clean_df.index <= next_time)]
    
    # Perform FFT analysis only if there is data in the sliced DataFrame
    n = len(sliced_df)
    if n > 0:
        t = sliced_df.index
        s = sliced_df['P1-Air-Supply-pressure']

        # FFT
        s -= s.mean()
        fr = np.fft.rfftfreq(n, dt)
        fou = np.fft.rfft(s)
        
        # Create a DataFrame for the current fr,amp
        temp_df = pd.DataFrame({'Frequency': fr,'Amplitude': np.abs(fou)})

        # Append the DataFrame to the list
        freq_amp_dfs.append(temp_df)
        
    else:
        # Handle the case where the DataFrame is empty (e.g., print a message)
        print(f"No data in the time range {current_time} - {next_time}")

    current_time = next_time    

In [ ]:
# Concatenate all DataFrames in the list into a single DataFrame
freq_amp_df = pd.concat(freq_amp_dfs, ignore_index=True)

In [ ]:
# Save the result DataFrame to a CSV file
freq_amp_df.to_csv(r"D:\\P1_Validation-3_Fourcrack\\Cleaned data_FFT_Df\\03.P1_22Oct23_fr,amp.csv")

In [ ]:
entropy_list = []

In [ ]:
current_time = start_time
while current_time <= end_time:
        
    next_time = current_time + time_interval
        
    # Slice the DataFrame based on the specified time range
    sliced_df = clean_df.iloc[(clean_df.index >= current_time) & (clean_df.index <= next_time)]
        
    # Perform FFT analysis only if there is data in the sliced DataFrame
    # Calculate the probabilities
    asp_prob =  sliced_df['P1-Air-Supply-pressure'].value_counts()/sum(sliced_df['P1-Air-Supply-pressure'].value_counts().values)
        
    # Calculate the entropy
    asp_entropy = sp.stats.entropy(asp_prob, base=2)
            
    # Create a DataFrame for the current fr,amp
    #temp_df = pd.DataFrame(asp_entropy,columns=['Entropy'])

    entropy_list.append(asp_entropy)
            
    current_time = next_time    

In [ ]:
Sh_En = pd.DataFrame(entropy_list,columns=['Entropy'])

In [ ]:
# Save the result DataFrame to a CSV file
Sh_En.to_csv(r"D:\\P1_Validation-3_Fourcrack\\Cleaned data_Shannon Enrtropy df\\03.P1_22Oct23_shannon_val.csv")